# 🏙️ Project 13 — Find Urban Areas

Excellent! In the previous project, you displayed **all land cover classes** from the ESA WorldCover dataset.

Now you'll focus on **just one class**:

> **Built-up (Urban) Areas**

Instead of visualizing forests, croplands, water, and cities together, you'll extract **only the urban pixels**.

This is your first project using a **categorical land cover dataset** to isolate a specific land cover type.

---

# 🏙️ Project 13 — Find Urban Areas

## Goal

Display only the **built-up (urban) areas** from the ESA WorldCover dataset.

Examples of built-up areas include:

- 🏢 Buildings
- 🏠 Residential neighborhoods
- 🛣 Roads
- 🏭 Industrial areas
- 🏙 Cities and towns

---

# Why is this Useful in Remote Sensing?

Urban maps are widely used for:

- 🏙 Urban expansion monitoring
- 🚧 City planning
- 🚗 Transportation studies
- 🌡 Urban Heat Island research
- 🌍 Population and environmental studies

For example, researchers can compare urban maps from different years to study how cities have grown over time.

---

# New Skill: Filtering a Specific Class

Until now:

```text
Land Cover Map
      ↓
Display All Classes
```

Today:

```text
Land Cover Map
      ↓
Select Built-up Class
      ↓
Urban Mask
      ↓
Display Urban Areas
```

This introduces an important idea:

> We don't always need every land cover class.

Sometimes we're interested in **only one category**.

---

# Remember from Project 12

The ESA WorldCover dataset stores **numbers**, not words.

For example:

| Value | Class |
|-------:|-------|
| 10 | Tree cover |
| 40 | Cropland |
| 50 | Built-up |
| 80 | Water |

Today's lesson focuses on:

```text
50 = Built-up
```

---

# Step 1 — Define the Study Area

We'll use the same study area.

```javascript
var countryArea = ee.Geometry.Polygon([
  [
    [44.00,25.00],
    [63.50,25.00],
    [63.50,39.80],
    [44.00,39.80],
    [44.00,25.00]
  ]
]);
```

Nothing new here.

The study area tells Earth Engine **where to perform the analysis**.

---

# Step 2 — Load the Land Cover Dataset

```javascript
var landCover = ee.ImageCollection(
  "ESA/WorldCover/v200"
)
.first()
.clip(countryArea);
```

Let's break it down.

`ee.ImageCollection()`

Loads the ESA WorldCover dataset.

↓

`.first()`

Selects the first image in the collection.

↓

`.clip(countryArea)`

Keeps only the pixels inside the study area.

After `.first()`, we have:

```text
ee.Image
```

which we can analyze.

---

# Step 3 — Select the Classification Band

```javascript
var landCoverClass = landCover.select('Map');
```

Remember:

The **Map** band stores the land cover class for every pixel.

Example:

```text
10
40
50
80
```

Each number represents a different surface type.

Unlike Sentinel-2 images, these values are **categories**, not reflectance measurements.

---

# Step 4 — Create an Urban Mask ⭐

This is today's new concept.

```javascript
var urbanMask = landCoverClass.eq(50);
```

Let's understand it.

You already know:

```javascript
.gt()
```

means:

```text
Greater Than
```

Today we use:

```javascript
.eq()
```

which means:

```text
Equal To
```

Earth Engine checks every pixel.

If the pixel value is:

```text
50
```

it becomes:

```text
1
```

Otherwise:

```text
0
```

Example:

Before:

```text
10
50
40
80
50
```

After:

```text
0
1
0
0
1
```

Now we have a **binary image** showing only urban areas.

---

# Step 5 — Display the Urban Areas

```javascript
Map.centerObject(countryArea,5);

Map.addLayer(
  urbanMask,
  {
    min:0,
    max:1,
    palette:['white','red']
  },
  'Urban Areas'
);
```

Interpretation:

- ⚪ White → Not urban
- 🔴 Red → Built-up areas

Zoom into a city like Tehran and you'll clearly see the built-up regions.

---

# Complete Code

```javascript
// 1. Study Area
var countryArea = ee.Geometry.Polygon([
  [
    [44.00,25.00],
    [63.50,25.00],
    [63.50,39.80],
    [44.00,39.80],
    [44.00,25.00]
  ]
]);

// 2. Load Dataset
var landCover = ee.ImageCollection(
  "ESA/WorldCover/v200"
)
.first()
.clip(countryArea);

// 3. Select Land Cover Band
var landCoverClass = landCover.select('Map');

// 4. Create Urban Mask
var urbanMask = landCoverClass.eq(50);

// 5. Display
Map.centerObject(countryArea,5);

Map.addLayer(
  urbanMask,
  {
    min:0,
    max:1,
    palette:['white','red']
  },
  'Urban Areas'
);
```

---

# Understanding `.eq()`

Think about each part:

```javascript
landCoverClass.eq(50)
```

```text
landCoverClass
```

↓

The classified land cover image.

↓

```text
.eq()
```

↓

Compare every pixel.

↓

```text
50
```

↓

Keep only pixels whose class value is **50 (Built-up)**.

The result is:

```text
Binary Image

0 = Not Urban

1 = Urban
```

---

# Visual Example

Before:

```text
10   40   50

80   50   10

40   50   80
```

After:

```text
0   0   1

0   1   0

0   1   0
```

Only the built-up pixels remain.

---

# Important Concepts to Remember

## 1. `.eq()`

`.eq()` means **Equal To**.

It compares every pixel with a specified value.

Example:

```javascript
landCoverClass.eq(50)
```

means:

> "Keep only pixels whose value is exactly **50**."

---

## 2. Binary Masks

Just like your water mask:

```text
Water

↓

0
1
1
0
```

Urban masks work the same way:

```text
Urban

↓

0
1
0
1
```

The only difference is **what the value represents**.

---

## 3. Land Cover Values Matter

Some important ESA WorldCover classes to remember:

| Value | Class |
|-------:|-------|
| 10 | Tree cover |
| 20 | Shrubland |
| 30 | Grassland |
| 40 | Cropland |
| 50 | Built-up |
| 60 | Bare / sparse vegetation |
| 70 | Snow and ice |
| 80 | Water |
| 90 | Wetlands |
| 95 | Mangroves |
| 100 | Moss and lichen |

You'll use these class values repeatedly in future projects.

---

## 4. Binary Masks Can Represent Different Things

Notice how the same idea appears in different projects.

For water:

```javascript
ndwi.gt(0)
```

For urban areas:

```javascript
landCoverClass.eq(50)
```

Both create **binary masks**, but they come from different sources:

- NDWI creates a mask using a mathematical threshold.
- Land cover creates a mask using a predefined class value.

---

# Common Beginner Mistakes

## ❌ Mistake 1: Using `.gt(50)` instead of `.eq(50)`

```javascript
landCoverClass.gt(50)
```

This selects **every class greater than 50**, not just urban areas.

Correct:

```javascript
landCoverClass.eq(50)
```

---

## ❌ Mistake 2: Applying `.eq(50)` to the Wrong Image

Use the **classification band**:

```javascript
landCoverClass.eq(50)
```

Not the original Sentinel-2 image.

Sentinel-2 pixels contain reflectance values, not land cover classes.

---

## ❌ Mistake 3: Forgetting What Class Number Represents Urban

Remember:

```text
50 = Built-up
```

Using the wrong class value will highlight an entirely different land cover type.

---

# Things to Remember

## 1. Land Cover Data is Categorical

Each pixel stores a **class**, not a reflectance value.

Example:

```text
50

↓

Built-up
```

not

```text
Reflectance = 50
```

---

## 2. `.eq()` Creates a Binary Mask

```javascript
landCoverClass.eq(50)
```

returns:

```text
1 = Urban

0 = Everything Else
```

---

## 3. Different Projects Use Different Rules

Water detection:

```javascript
ndwi.gt(0)
```

Urban detection:

```javascript
landCoverClass.eq(50)
```

Vegetation detection:

```javascript
ndvi.gt(0.5)
```

The pattern is the same:

```text
Image

↓

Apply Rule

↓

Binary Mask
```

---

## 4. Classification Values are Meaningful

Unlike NDVI values, land cover class numbers are **labels**.

For example:

```text
50

↓

Built-up
```

The number itself has **no mathematical meaning**.

It simply identifies a category.

---

# Your Practice (Project 13.1)

Create an urban map for your own study area.

## Requirements

1. Create a study area.
2. Load the ESA WorldCover dataset.
3. Select the **Map** band.
4. Create an urban mask using `.eq(50)`.
5. Display the urban areas.

---

Before writing the code, answer these two questions.

## Question 1

Why do we use:

```javascript
.eq(50)
```

instead of:

```javascript
.gt(50)
```

to find urban areas?

---

## Question 2

Both of these produce a binary image:

```javascript
ndwi.gt(0)
```

and

```javascript
landCoverClass.eq(50)
```

What is the difference between them?

Send me your answers and your code, and I'll review both your understanding and your implementation exactly like a university professor.